# Research Paper Search & Analysis Engine
A semantic + keyword search system over ML ArXiv papers, with clustering, keyword extraction, and automatic summarization.


## 1. Install dependencies
`transformers` is pinned below **`5.0`** on purpose — version 5.0 (Jan 2026) removed the classic `summarization` pipeline task, which this project relies on.

In [ ]:
!pip install -q datasets "transformers<5.0" sentencepiece accelerate faiss-cpu sentence-transformers keybert


## 2. Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from transformers import pipeline
from keybert import KeyBERT
import faiss


## 3. Load the dataset

In [ ]:
dataset = load_dataset("CShorten/ML-ArXiv-Papers", split="train")
print(dataset)


In [ ]:
df = pd.DataFrame(dataset)
df = df[["title", "abstract"]].dropna().reset_index(drop=True)
print(df.shape)
df.head()


## 4. Build a working sample
The full dataset has ~117k rows. We use a random sample of 3000 papers so embeddings, clustering, and summarization stay fast.

In [ ]:
df = df.sample(3000, random_state=42).reset_index(drop=True)

df["paper_text"] = (df["title"] + " " + df["abstract"]).str.replace("\n", " ", regex=False).str.strip()

print(df.shape)
df[["title", "paper_text"]].head()


## 5. Semantic embeddings (Sentence-Transformers)

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")

EMB_PATH = "paper_embeddings.npy"

if os.path.exists(EMB_PATH):
    os.remove(EMB_PATH)

print("Generating embeddings...")
embeddings = model.encode(
    df["paper_text"].tolist(),
    batch_size=128,
    show_progress_bar=True,
    convert_to_numpy=True,
).astype("float32")

np.save(EMB_PATH, embeddings)
print("Embeddings shape:", embeddings.shape)


## 6. Build the FAISS similarity index

In [ ]:
faiss.normalize_L2(embeddings)

dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

faiss.write_index(index, "paper_faiss.index")
print("FAISS index built. Total vectors:", index.ntotal)


## 7. Semantic search function

In [ ]:
def semantic_search(query, top_k=5):
    """Return the top_k most semantically similar papers to `query` as a DataFrame."""
    query_embedding = model.encode([query], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(query_embedding)

    D, I = index.search(query_embedding, top_k)

    results = df.iloc[I[0]][["title", "abstract"]].copy()
    results["similarity_score"] = D[0]
    return results.reset_index(drop=True)


In [ ]:
semantic_search("deep learning for medical image analysis")


## 8. TF-IDF keyword-based search (for comparison)

In [ ]:
tfidf = TfidfVectorizer(stop_words="english")
tfidf_matrix = tfidf.fit_transform(df["paper_text"])

def tfidf_search(query, top_k=5):
    """Return the top_k papers by TF-IDF cosine similarity to `query` as a DataFrame."""
    query_vector = tfidf.transform([query])
    similarity = cosine_similarity(query_vector, tfidf_matrix).flatten()

    top_indices = similarity.argsort()[-top_k:][::-1]

    results = df.iloc[top_indices][["title", "abstract"]].copy()
    results["similarity_score"] = similarity[top_indices]
    return results.reset_index(drop=True)


In [ ]:
tfidf_search("deep learning for medical image analysis")


## 9. Compare TF-IDF vs. semantic search side by side

In [ ]:
query = "Machine Learning in Healthcare"

print("TF-IDF Results")
display(tfidf_search(query))

print("\nSemantic Search Results")
display(semantic_search(query))


## 10. Clustering research papers with KMeans

In [ ]:
kmeans = KMeans(n_clusters=10, random_state=42, n_init=10)
df["cluster"] = kmeans.fit_predict(embeddings)

df["cluster"].value_counts().sort_index()


In [ ]:
df[df["cluster"] == 0][["title"]].head(10)


## 11. Visualizing clusters with PCA

In [ ]:
pca = PCA(n_components=2, random_state=42)
reduced = pca.fit_transform(embeddings)

plt.figure(figsize=(10, 7))
plt.scatter(reduced[:, 0], reduced[:, 1], c=df["cluster"], cmap="tab10", s=15)
plt.title("Research Paper Clusters")
plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.colorbar(label="Cluster")
plt.show()


## 12. Keyword extraction with KeyBERT

In [ ]:
kw_model = KeyBERT()

paper = df.iloc[0]["paper_text"]
keywords = kw_model.extract_keywords(
    paper,
    keyphrase_ngram_range=(1, 2),
    stop_words="english",
    top_n=8,
)
keywords


## 13. Abstractive summarization
Uses the classic `summarization` pipeline task (BART), which is why `transformers` is pinned below version 5.0 in step 1.

In [ ]:
summarizer = pipeline(
    task="summarization",
    model="facebook/bart-large-cnn",
)


In [ ]:
def summarize_abstract(text, max_length=120, min_length=40):
    summary = summarizer(text, max_length=max_length, min_length=min_length)
    return summary[0]["summary_text"]

sample_abstract = df.iloc[0]["abstract"]
print(summarize_abstract(sample_abstract))


## 14. Put it all together: search + summarize

In [ ]:
def search_and_summarize(query, top_k=5):
    results = semantic_search(query, top_k=top_k)

    for _, row in results.iterrows():
        print("Title:", row["title"])
        print(f"Similarity: {row['similarity_score']:.4f}")
        print("Summary:", summarize_abstract(row["abstract"]))
        print("-" * 80)

search_and_summarize("deep learning for medical image analysis", top_k=3)
